# Music Gen Pipeline — Colab T4

Run cells top to bottom. Runtime → Change runtime type → **T4 GPU**.

In [ ]:
# Cell 1: Setup — clone repo and install dependencies
import os

REPO_URL = "https://github.com/Aryan9403/soundpaint.git"  # TODO: update
REPO_DIR = "/content/soundpaint"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# Install dependencies (mamba_ssm requires CUDA build)
!pip install -q torch>=2.0.0 descript-audio-codec audiotools librosa soundfile numpy pyyaml tqdm
!pip install -q mamba_ssm causal-conv1d

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Upload dataset
# Option A: Upload a zip of your dataset/
from google.colab import files
import zipfile, os

uploaded = files.upload()  # select your dataset.zip

for fname in uploaded:
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('.')
        print(f"Extracted {fname}")

# Check files
import glob
mp3s = glob.glob('dataset/**/*.mp3', recursive=True)
print(f"Found {len(mp3s)} .mp3 files")

In [ ]:
# Cell 3: Tokenize audio with DAC
# This takes ~2-5 min on T4 for 240 files
!python data/prepare.py \
    --audio_dir dataset/ \
    --output_dir data/tokens_small \
    --device cuda \
    --duration 30

# Verify
import torch, glob
pt_files = glob.glob('data/tokens_small/*.pt')
print(f"Token files: {len(pt_files)}")
if pt_files:
    sample = torch.load(pt_files[0], weights_only=True)
    print(f"Token shape: {sample.shape} | dtype: {sample.dtype}")

In [ ]:
# Cell 4: Train
# ~30-60 min for 2000 steps on T4
!python train.py --config configs/tiny.yaml

# Check checkpoints
import glob
ckpts = glob.glob('checkpoints/tiny/*.pt')
print(f"Checkpoints saved: {ckpts}")

In [ ]:
# Cell 5: Generate audio
!python generate.py \
    --checkpoint checkpoints/tiny/best.pt \
    --duration 10 \
    --output output.wav

import os
print(f"Output size: {os.path.getsize('output.wav') / 1024:.1f} KB")

In [ ]:
# Cell 6: Listen to generated audio in-notebook
from IPython.display import Audio, display

display(Audio('output.wav'))

# Also download it
from google.colab import files
files.download('output.wav')